# Session 7: Taking noise away, and what all this was for

**259.075 Data-integrated Algorithmic Design Processes II** · Session 7 of 7

Session 5 drew shapes from a distribution, session 6 drew letters. Both were the same principle
with different outputs. The image models everybody talks about work differently, and the
difference is worth half an hour, because once you have seen it you will never again wonder what
the "steps" slider in an image tool does.

After that we go back to session 1 and give the thing we have been building its proper name.

In [ ]:
import sys
import numpy as np
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.insert(0, "../data")
from vae import VAE
import diffusion as D

bilder = np.load("../data/grundrisse.npy")
codes = np.load("../data/vae_codes.npz")["code"]
Z = torch.tensor(codes, dtype=torch.float32)

vae = VAE()
vae.load_state_dict(torch.load("../data/vae_8.pt"))
vae.eval()

modell = D.Rauschschaetzer()
modell.load_state_dict(torch.load("../data/diffusion.pt"))
modell.eval()

alpha_quer = D.plan()
print(f"{len(Z)} codes of {Z.shape[1]} numbers, {D.SCHRITTE} steps on the noise ladder")


def zeige(stapel, titel, spalten=8, hoehe=120):
    zeilen = int(np.ceil(len(stapel) / spalten))
    fig = make_subplots(rows=zeilen, cols=spalten, horizontal_spacing=0.01,
                        vertical_spacing=0.04)
    for n, bild in enumerate(stapel):
        r, c = divmod(n, spalten)
        fig.add_trace(go.Heatmap(z=bild[::-1], colorscale="Greys", reversescale=True,
                                 showscale=False), row=r + 1, col=c + 1)
    fig.update_layout(height=hoehe * zeilen + 80, title=titel)
    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)
    return fig


def male(z):
    with torch.no_grad():
        return torch.sigmoid(vae.decodiere(z)).numpy().reshape(-1, 32, 32)

## Forwards is free

Take a building and add a little noise. Add a little more. Keep going, two hundred times, and at
the end there is nothing left but noise. That direction needs no model at all, only a random
number generator, and you can jump straight to step 137 without computing the first 136.

That is the forward process, and the whole idea of a diffusion model is the question: can the
walk be reversed?

In [ ]:
haus = 12
z0 = Z[haus:haus + 1]
stufen = [0, 20, 50, 90, 140, 200]
verrauscht = []
wuerfel = torch.Generator().manual_seed(3)
rauschen = torch.randn(1, D.ENGE, generator=wuerfel)
for t in stufen:
    a = alpha_quer[t]
    verrauscht.append(a.sqrt() * z0 + (1 - a).sqrt() * rauschen)

zeige(np.concatenate([male(v) for v in verrauscht]),
      "one building, drowned in noise step by step", spalten=len(stufen)).show()
print("signal left at each step:", [f"{float(alpha_quer[t]):.3f}" for t in stufen])

## Backwards costs a model

Here is the surprise: the model does not learn to make buildings. It learns to look at a noisy
thing, be told how noisy it is, and say which part of it was the noise. That is an ordinary
regression with a squared error, exactly the job from session 1, only with eight outputs instead
of one.

Everything else is bookkeeping. Once you can estimate the noise, you can subtract it and get an
estimate of the clean thing. Then you add back a little less noise than you took off, step down
the ladder, and repeat. Write the subtraction.

> The model never sees a whole building being built. It only ever answers: how much of this is
> noise?

One more thing about where this runs. Our images are 32 by 32, which is 1024 numbers, and
running two hundred denoising steps on 1024 numbers without a graphics card is not a lecture,
it is a coffee break. So the diffusion happens on the eight numbers from session 5 instead, and
the decoder turns the result into a picture at the very end. That is not a shortcut invented for
this course: it is what the word *latent* does in "latent diffusion", and it is the reason those
models fit on a consumer card at all.

In [ ]:
def entrausche(modell, zt, t, alpha_quer):
    """One step: what does the model think the clean code was?"""

    # YOUR TURN: two lines. Estimate the noise in zt with the model, then undo the forward
    # step: z0 = (zt - sqrt(1 - a) * noise) / sqrt(a), with a = alpha_quer[t].
    raise NotImplementedError("denoising step missing")

    return z0.clamp(-4.0, 4.0)


# a quick check against the ladder above: at step 20 the estimate should be close to the
# building we started from, at step 200 it should not be
for t in (20, 200):
    a = alpha_quer[t]
    zt = a.sqrt() * z0 + (1 - a).sqrt() * rauschen
    schaetzung = entrausche(modell, zt, t, alpha_quer).detach()
    print(f"step {t:3d}: distance to the true code {float((schaetzung - z0).norm()):.2f}")

In [ ]:
zeige(np.concatenate([male(entrausche(modell,
                                      alpha_quer[t].sqrt() * z0
                                      + (1 - alpha_quer[t]).sqrt() * rauschen,
                                      t, alpha_quer))
                      for t in stufen]),
      "what the model guesses the building was, from each of the six noise levels",
      spalten=len(stufen)).show()

Close to the clean end it recovers the building almost exactly, because there is barely anything
to undo. From pure noise it produces something that is a building, just not that one, which is
the entire trick: at the top of the ladder the answer is unconstrained, so the walk downwards
invents.

## The steps slider

Generating means walking the ladder from top to bottom. You do not have to visit every rung.
The number of rungs you do visit, each one a call to the model, is the setting every image tool
calls *steps*; fewer steps means skipping more rungs.

In [ ]:
schrittzahlen = [2, 5, 10, 25, 50, 200]
gezogen = {s: male(D.ziehe(modell, alpha_quer, n=8, schritte=s,
                           wuerfel=torch.Generator().manual_seed(0)))
           for s in schrittzahlen}

fig = make_subplots(rows=1, cols=8, horizontal_spacing=0.01)
for n in range(8):
    fig.add_trace(go.Heatmap(z=gezogen[200][n][::-1], colorscale="Greys", reversescale=True,
                             showscale=False), row=1, col=n + 1)
fig.frames = [go.Frame(name=str(s),
                       data=[go.Heatmap(z=gezogen[s][n][::-1], colorscale="Greys",
                                        reversescale=True, showscale=False)
                             for n in range(8)],
                       traces=list(range(8)),
                       layout=go.Layout(title=f"{s} steps"))
              for s in schrittzahlen]
fig.update_layout(height=230, title="200 steps",
                  sliders=[dict(active=len(schrittzahlen) - 1, currentvalue=dict(prefix="steps "),
                                steps=[dict(method="animate", label=str(s),
                                            args=[[str(s)],
                                                  dict(mode="immediate",
                                                       frame=dict(duration=0, redraw=True))])
                                       for s in schrittzahlen])])
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.show()

for s in schrittzahlen:
    z = D.ziehe(modell, alpha_quer, n=64, schritte=s, wuerfel=torch.Generator().manual_seed(1))
    print(f"{s:3d} steps: spread of the drawn codes {float(z.std()):.2f} "
          f"(real codes: {float(Z.std()):.2f})")

Two steps is not enough: the codes come out more than twice as spread as the real ones, and the
decoder makes fragments of them. By five the spread has already settled, and from there more
steps buy tidiness rather than correctness. Somewhere around twenty-five the pictures stop
changing in any way you could defend in a jury.

That is the trade you are making every time you push that slider up: linear cost, and a benefit
that stops.

## LoRA

Now a practical problem. Say you want this model to produce narrow buildings, the long thin
Vienna side wings. You could train the whole thing again on narrow buildings only, and for a
model of this size that costs a minute. For a real image model it costs a data centre, and you
would have a second complete model to store.

The trick everybody uses instead: freeze the model. Next to a weight matrix, add two thin
matrices whose product has the same shape, and train only those. The input layer here is 72 by
256, which is 18,432 numbers; two thin ones with four columns in between come to 1,312. The small
pair is called a *low-rank adaptation*, LoRA, and it can be plugged in and pulled out like a key.

Narrow is approximated here by the quarter of buildings with the lowest fill: long thin wings
fill little of their frame, though so do outlines with courtyards.

In [ ]:
schlank = np.load("../data/lora_auswahl.npy")
print(f"the LoRA was trained on {int(schlank.sum())} buildings, the quarter with the lowest fill")

mit_lora = D.Rauschschaetzer()
mit_lora.load_state_dict(torch.load("../data/diffusion.pt"))
mit_lora.ein = D.LoRA(mit_lora.ein)
mit_lora.aus[1] = D.LoRA(mit_lora.aus[1])
mit_lora.load_state_dict(torch.load("../data/lora_schlank.pt"), strict=False)
mit_lora.eval()

grund = sum(p.numel() for p in modell.parameters())
zusatz = sum(p.numel() for n, p in mit_lora.named_parameters()
             if n.endswith(".a") or n.endswith(".b"))
print(f"base model: {grund:,} parameters")
print(f"lora      : {zusatz:,} parameters, {100 * zusatz / grund:.1f}% of the base model")

fig = go.Figure(go.Bar(x=["base model", "lora"], y=[grund, zusatz],
                       marker_color=["#4c78a8", "#d8a24a"], text=[f"{grund:,}", f"{zusatz:,}"]))
fig.update_layout(height=320, yaxis_type="log", yaxis_title="parameters",
                  title="what gets trained when you train a LoRA")
fig.show()

In [ ]:
w = torch.Generator().manual_seed(11)
ohne = male(D.ziehe(modell, alpha_quer, n=8, schritte=50,
                    wuerfel=torch.Generator().manual_seed(11)))
mit = male(D.ziehe(mit_lora, alpha_quer, n=8, schritte=50,
                   wuerfel=torch.Generator().manual_seed(11)))

fig = make_subplots(rows=2, cols=8, horizontal_spacing=0.01, vertical_spacing=0.08,
                    row_titles=["base model", "with LoRA"])
for zeile, stapel in enumerate([ohne, mit], start=1):
    for spalte, bild in enumerate(stapel, start=1):
        fig.add_trace(go.Heatmap(z=bild[::-1], colorscale="Greys", reversescale=True,
                                 showscale=False), row=zeile, col=spalte)
fig.update_layout(height=340, title="same model, same random start, LoRA plugged in below")
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.show()

print(f"average fill, base model: {(ohne > 0.5).mean():.3f}")
print(f"average fill, with lora : {(mit > 0.5).mean():.3f}")
print(f"the low-fill quarter it was trained on: {bilder[schlank].mean():.3f}")
print(f"all buildings                       : {bilder.mean():.3f}")

The strength is a dial too. The LoRA contributes its product multiplied by a number, and that
number does not have to be one: at zero you have the base model back, at a half you get something
in between, and beyond one you are pushing the correction further than it was trained for. Watch
where it stops helping and starts eating the buildings. If you have ever wondered what the weight
slider next to a LoRA in an image tool does, this is it.

In [ ]:
staerken = [0.0, 0.5, 1.0, 2.0, 3.0]
reihen = []
for s in staerken:
    mit_lora.ein.staerke = s
    mit_lora.aus[1].staerke = s
    reihen.append(male(D.ziehe(mit_lora, alpha_quer, n=6, schritte=50,
                               wuerfel=torch.Generator().manual_seed(11))))
mit_lora.ein.staerke = mit_lora.aus[1].staerke = 1.0

fig = make_subplots(rows=len(staerken), cols=6, horizontal_spacing=0.01, vertical_spacing=0.03,
                    row_titles=[f"× {s}" for s in staerken])
for zeile, stapel in enumerate(reihen, start=1):
    for spalte, bild in enumerate(stapel, start=1):
        fig.add_trace(go.Heatmap(z=bild[::-1], colorscale="Greys", reversescale=True,
                                 showscale=False), row=zeile, col=spalte)
fig.update_layout(height=110 * len(staerken) + 80, title="how hard the LoRA is pushed")
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.show()

for s, stapel in zip(staerken, reihen):
    print(f"strength {s}: average fill {(stapel > 0.5).mean():.3f}")

## Three words that get confused

*Pretraining* is what produced the base model: all the data, all the weights, once, expensively.

*Fine-tuning* is adjusting a finished model on your own, much smaller data. A LoRA is the cheap
version of it, where you adjust a few thousand extra weights instead of all of them and can
switch the result off again.

A *prompt* changes nothing inside the model at all. It supplies context: a language model
continues it, an image model uses it to steer the denoising. When somebody says they "trained" a model by typing instructions at it, they
did not.

> Prompting is the one you do every day, and it is the only one of the three that changes no
> weight at all.

## What we have been building

Session 1 opened with a complaint: entering a building into a simulation takes an afternoon, and
a hundred design variants take a month. Everything since has been one answer to that, and it has
a name in the literature. A *surrogate model* is a fast model that imitates a slow one, and that
is what you built in session 3: a network that answers in microseconds where a simulation takes
minutes.

The rest of the course was about the two things that make such a model usable. Session 4 was
about what you feed it, because a number somebody chose is already an argument about what
matters. Sessions 5 to 7 were about generating rather than predicting, which is the same
machinery pointed the other way.

What stays with you, if nothing else does, are three questions to ask of any model somebody
hands you, including your own:

1. Where does the data come from, and what does the target actually measure? Ours was computed
   from a formula, and we said so on every page.
2. What was held out, and when was it looked at? A number from a test set that was consulted
   during development is not a test number.
3. What is the range it was checked on? Session 3 broke a polynomial with two buildings that
   were slightly too compact. Nothing in an output warns you that you have left the range.

Ask those three about a consultant's energy model and you will either get a good answer or watch
the conversation change.

## Tests

In [ ]:
# the denoising step has to undo the forward step exactly when the true noise is known
class Orakel(torch.nn.Module):
    def forward(self, z, t):
        return rauschen.expand(len(z), -1)


for t in (5, 60, 199):
    a = alpha_quer[t]
    zt = a.sqrt() * z0 + (1 - a).sqrt() * rauschen
    zurueck = entrausche(Orakel(), zt, t, alpha_quer)
    assert torch.allclose(zurueck, z0, atol=1e-3), (
        f"with the true noise, step {t} has to give the original code back")

# with the real model, an almost clean code has to come back almost unchanged
a = alpha_quer[5]
nah = entrausche(modell, a.sqrt() * z0 + (1 - a).sqrt() * rauschen, 5, alpha_quer)
weit = entrausche(modell, alpha_quer[200].sqrt() * z0
                  + (1 - alpha_quer[200]).sqrt() * rauschen, 200, alpha_quer)
assert float((nah - z0).detach().norm()) < float((weit - z0).detach().norm()), (
    "the estimate from a barely noisy code has to be closer than the one from pure noise")

# generating has to land in the region the real codes occupy
gez = D.ziehe(modell, alpha_quer, n=64, schritte=50, wuerfel=torch.Generator().manual_seed(2))
assert abs(float(gez.std()) - float(Z.std())) < 0.4, (
    f"drawn codes spread {float(gez.std()):.2f} against real {float(Z.std()):.2f}")

# the LoRA has to change something, and switching it off has to change it back
mit_lora.ein.staerke = mit_lora.aus[1].staerke = 0.0
aus = D.ziehe(mit_lora, alpha_quer, n=8, schritte=50, wuerfel=torch.Generator().manual_seed(11))
mit_lora.ein.staerke = mit_lora.aus[1].staerke = 1.0
an = D.ziehe(mit_lora, alpha_quer, n=8, schritte=50, wuerfel=torch.Generator().manual_seed(11))
basis = D.ziehe(modell, alpha_quer, n=8, schritte=50, wuerfel=torch.Generator().manual_seed(11))
assert torch.allclose(aus, basis, atol=1e-4), "strength 0 has to give the base model back"
assert not torch.allclose(an, basis, atol=1e-2), "strength 1 has to change something"

print("all green")

## Where you stand now

You can say what a diffusion model learns, which is smaller than most people assume, and why
generating costs a walk down a ladder while adding noise is free. You know what the steps slider
buys and where it stops buying. You can explain LoRA in one sentence and back it with the
parameter count. And you can tell pretraining, fine-tuning and prompting apart, which is more
than most of the people who will be selling you these tools.

Seven sessions ago this was one neuron and a straight line through a cloud of Viennese
buildings. Nothing since has been fundamentally harder; it has been the same parts, stacked,
pointed in different directions, and trained on different things.

---

### Terms to learn

| term | in one sentence |
|---|---|
| diffusion model | learns to estimate the noise in a noisy input; generates by removing it step by step |
| forward process | adding noise; needs no model and can jump to any step |
| denoising step | one rung down the ladder, using the model's noise estimate |
| steps | how many rungs you actually visit; linear cost, diminishing returns |
| latent diffusion | running the diffusion on codes instead of pixels, which is why it fits on a normal machine |
| LoRA | two thin matrices trained beside a frozen one; a small, removable adaptation |
| LoRA strength | how hard the adaptation is applied; zero gives the base model back |
| pretraining | building the base model from all the data, once, expensively |
| fine-tuning | adjusting a finished model on your own smaller data |
| surrogate model | a fast model imitating a slow simulation, valid only where it was checked |

## Homework

Ten points, due before the next session: four for running it, six for your thoughts on it.

**How to do it**

1. Fill in the `YOUR TURN` gap above, yourself or from the solution notebook. If you took it from
   the solution, say so in one line at the top of your answers; that costs nothing.
2. Below, change only the lines under `# YOUR CHOICE`.
3. Write your answers into the last cell.
4. Choose Kernel, Restart Kernel and Run All Cells, check that everything runs, and upload the
   notebook in TUWEL.

**How it is graded**

- Four points for the run: it runs without an error (one), it shows what the task asks for (one),
  and every number you give about your run appears in your output or is worked out from it in a
  step you write down (two). Numbers the questions give you, examples from outside the notebook
  and forecasts you mark as your own expectation are exempt.
- Six points for your answers, one per question, no halves. Say what you saw and what it means,
  and you get the point, also with borrowed code. A sentence that would fit any model gets none.

### The task

1. Choose a subset of the Vienna outlines by a condition of your own on the table. Before you
   train, write one sentence on what you expect the LoRA to do to the drawn shapes; that
   sentence goes into your answer to question 2.
2. Train your LoRA on the subset and draw with and without it from the same random start. Show
   both.
3. In your text, say whether the difference is visible, how you tell, and what you would change
   about the way you chose the subset.
4. The sixth question is about the whole course: the network chosen in class in session 3,
   and the three questions from above, answered with numbers from that session.

In [ ]:
tabelle = np.genfromtxt("../data/grundrisse.csv", delimiter=",", names=True,
                        dtype=None, encoding="utf-8")

# YOUR CHOICE: a condition on the table, e.g. tabelle["rundheit"] > 0.6. Columns: startjahr,
# flaeche_m2, kompaktheit, rundheit, fuellgrad. Two conditions together need brackets and &:
# (tabelle["startjahr"] >= 1900) & (tabelle["startjahr"] < 1945). A hundred buildings or more
# is a practical minimum here; with fewer the LoRA is more likely to learn the examples than
# the kind.
meine_auswahl = None

assert meine_auswahl is not None, "enter a condition of your own"
meine_auswahl = np.asarray(meine_auswahl)
assert meine_auswahl.dtype == bool and meine_auswahl.shape == (len(tabelle),), (
    "the selection has to be a condition on the table, true or false for each of its rows")
print(f"{int(meine_auswahl.sum())} of {len(tabelle)} buildings chosen")
assert meine_auswahl.sum() >= 100, "fewer than a hundred: widen the condition"
assert meine_auswahl.sum() < len(tabelle), "that is every building; choose a real subset"


def quadratisch(fig):
    for name in fig.layout:
        if name.startswith("yaxis"):
            fig.layout[name].scaleanchor = "x" + name[5:]
    return fig


quadratisch(zeige(bilder[meine_auswahl][:12], "twelve of the buildings you chose", spalten=12,
                  hoehe=150)).show()

In [ ]:
# YOUR CHOICE: a whole number as the seed for training and drawing
samen_h = None

assert isinstance(samen_h, int), "enter a whole number as the seed"
torch.manual_seed(samen_h)
meins = D.Rauschschaetzer()
meins.load_state_dict(torch.load("../data/diffusion.pt"))
for p in meins.parameters():
    p.requires_grad_(False)
meins.ein = D.LoRA(meins.ein)
meins.aus[1] = D.LoRA(meins.aus[1])
zusatz_h = [p for p in meins.parameters() if p.requires_grad]
D.trainiere(Z[torch.tensor(meine_auswahl)], alpha_quer, samen=samen_h, epochen=1500,
            modell=meins, nur=zusatz_h, lernrate=1e-3)
meins.eval()


def breite_zu_hoehe(stapel):
    # per outline: width over height of the box around its filled cells, then the average
    werte = []
    for bild in stapel > 0.5:
        zeilen, spalten = np.nonzero(bild.any(axis=1))[0], np.nonzero(bild.any(axis=0))[0]
        if len(zeilen):
            werte.append((spalten[-1] - spalten[0] + 1) / (zeilen[-1] - zeilen[0] + 1))
    return float(np.mean(werte))


ohne_h = male(D.ziehe(modell, alpha_quer, n=12, schritte=50,
                      wuerfel=torch.Generator().manual_seed(samen_h)))
mit_h = male(D.ziehe(meins, alpha_quer, n=12, schritte=50,
                     wuerfel=torch.Generator().manual_seed(samen_h)))
quadratisch(zeige(np.concatenate([ohne_h, mit_h]),
                  "top: base model; bottom: with your LoRA; same start in each column",
                  spalten=12, hoehe=150)).show()

for name, stapel in (("base model", ohne_h), ("with your lora", mit_h),
                     ("your subset", bilder[meine_auswahl]), ("all buildings", bilder)):
    print(f"{name:15s} fill {(stapel > 0.5).mean():.3f}, "
          f"width to height of the bounding box {breite_zu_hoehe(stapel):.2f}")
print(f"\nyour LoRA trained {sum(p.numel() for p in zusatz_h):,} numbers "
      f"on {int(meine_auswahl.sum())} buildings")

### Your interpretation

1. **Size.** By how much does the LoRA move the fill and the width-to-height ratio, compared with how far your subset differs from all buildings?
2. **Plausibility.** Your sentence written before training, then: is a difference visible by eye, and does it go the way you expected? Point to columns in the figure. "No visible difference" or "the opposite direction" is a valid answer with evidence.
3. **Units.** Every outline is scaled to fill its frame. Is the property you chose by visible on the grid at all, directly or through shapes that go with it? Say what that means for what the LoRA could learn.
4. **Range.** How many buildings did your LoRA learn from, against how many numbers it trained? Say what you would expect with half as many buildings.
5. **The subset.** What would you change about the way you chose it, and why would it help?
6. **The course.** The network chosen in class in session 3, and the three questions, one short paragraph each, with numbers from that session. Where did its data come from and what did the target measure? What was held out, and when was it looked at? On which range was it trained, and on which range was it checked?

*Your text here.*